# Parameter recovery

A fit reports parameters. Recovery asks a prior question: if you know the parameters that produced the data, does the fit give them back?

Simulate, then fit, then compare. You choose the parameters. You make the data. You fit the model as if you had not chosen them. Then you look at the distance between the two sets of parameters.


## When the data sit on the line

The line is $y = 1 + 2x$, with no noise. For $x = 1, 2, 3$ the scores are 3, 5, and 7. Least squares, as in the regression lesson, returns an intercept and a slope. On these points there is nothing besides that line for the fit to follow.


In [ ]:
# Least-squares intercept and slope for a set of points. The totals below are the pieces of that formula.
# Fit one line and hand back its intercept and its slope.
def fit_line(xs, ys):
    # How many points.
    n = len(xs)
    # Running totals for the least-squares formulas.
    sum_x = 0
    sum_y = 0
    sum_xy = 0
    sum_xx = 0
    # Add each point into those totals.
    for i in range(n):
        sum_x = sum_x + xs[i]
        sum_y = sum_y + ys[i]
        sum_xy = sum_xy + xs[i] * ys[i]
        sum_xx = sum_xx + xs[i] ** 2
    # The slope, from the totals.
    slope = (n * sum_xy - sum_x * sum_y) / (n * sum_xx - sum_x ** 2)
    # The intercept, from the two means and that slope.
    intercept = sum_y / n - slope * (sum_x / n)
    return intercept, slope

# These outcomes sit on intercept 1 and slope 2, with nothing added, so the fit should hand those two values back.
# Three inputs that sit on a known line.
xs = [1, 2, 3]
# The outcomes, filled by the loop.
ys = []
# Each outcome is 1 plus 2 times x.
for x in xs:
    ys.append(1 + 2 * x)
# The outcomes, then the intercept and slope the fit recovers.
print(ys)
print(fit_line(xs, ys))


## When one score moves

Replace the last score with 8. The parameters that made the original line are still 1 and 2. The data are no longer that line. The fit follows the data it is given.


In [ ]:
# One score has moved off the line, so the recovered line moves too. The rule is the same fit as the cell above.
# The same fit, repeated so this cell can run on its own.
def fit_line(xs, ys):
    # How many points.
    n = len(xs)
    # Running totals for the formula.
    sum_x = 0
    sum_y = 0
    sum_xy = 0
    sum_xx = 0
    # Add each point into those totals.
    for i in range(n):
        sum_x = sum_x + xs[i]
        sum_y = sum_y + ys[i]
        sum_xy = sum_xy + xs[i] * ys[i]
        sum_xx = sum_xx + xs[i] ** 2
    # The slope.
    slope = (n * sum_xy - sum_x * sum_y) / (n * sum_xx - sum_x ** 2)
    # The intercept.
    intercept = sum_y / n - slope * (sum_x / n)
    return intercept, slope

# Points that are no longer exactly on intercept 1 and slope 2.
intercept, slope = fit_line([1, 2, 3], [3, 5, 8])
# The recovered intercept and slope, to three decimals.
print(round(intercept, 3), round(slope, 3))


The recovered slope is not 2. Recovery matched on the noiseless data. Once a score had moved, the fit did not return the parameters that had made the original line. A fit reports a line for the data in front of it. It does not, by itself, certify the parameters that generated them.


## The spread across repeats

Each frame draws 20 samples from $y = 1 + 2x$, plus noise, and recovers the slope. The grey line is the slope that made the samples, 2. A small sample scatters. A larger sample sits closer to that line.

Drag the slider under the figure, or press Play. The first time this session imports numpy, that library can take a moment to download.


In [ ]:
# Recovered slopes from repeated samples. The true slope is the horizontal line. Later frames use a larger sample.
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

# Fit that many samples of the given size, and collect the slope from each.
def recovered_slopes(n, repeats, rng):
    # One recovered slope per sample.
    slopes = []
    # A new sample on every pass.
    for i in range(repeats):
        # Inputs 1 through n.
        x = np.arange(1, n + 1, dtype=float)
        # The line 1 + 2x, plus a disturbance on each point.
        y = 1 + 2 * x + rng.normal(0, 1, size=n)
        # Means used by the slope formula.
        x_mean = x.mean()
        y_mean = y.mean()
        # The least-squares slope of this sample.
        slope = np.sum((x - x_mean) * (y - y_mean)) / np.sum((x - x_mean) ** 2)
        slopes.append(float(slope))
    return slopes

# A fixed start, so the cloud of slopes repeats.
rng = np.random.default_rng(0)
# Sample sizes, one frame each, from small to larger.
sizes = [4, 6, 8, 12, 16, 24]
# How many slopes to recover at each sample size.
repeats = 20
# The list of slopes for each sample size.
samples = []
# Recover a cloud of slopes at this sample size.
for n in sizes:
    samples.append(recovered_slopes(n, repeats, rng))

# Vertical limits with a little room around the most extreme slopes.
low = min(min(sample) for sample in samples) - 0.15
high = max(max(sample) for sample in samples) + 0.15

# The figure the player draws into.
fig, ax = plt.subplots(figsize=(6.4, 3.4), dpi=80)
# The true slope, as a horizontal line.
ax.axhline(2, color="#cccccc", linewidth=1)
# The recovered slopes, starting with none drawn.
dots = ax.scatter([], [], s=18)
# A narrow horizontal range. The dots are jittered so they do not pile up.
ax.set_xlim(-0.55, 0.55)
ax.set_ylim(low, high)
ax.set_xticks([])
# Name the vertical axis.
ax.set_ylabel("Recovered slope")

# One sample size per frame.
def draw(frame):
    # The slopes recovered at this sample size.
    slopes = samples[frame]
    # A small horizontal offset so identical slopes stay visible.
    jitter = []
    for i in range(len(slopes)):
        jitter.append((i - (len(slopes) - 1) / 2) * 0.03)
    # Place each slope at its jittered horizontal position.
    dots.set_offsets(np.column_stack([jitter, slopes]))
    # Title the frame with the sample size.
    ax.set_title("n = " + str(sizes[frame]))
    return dots,

# Each frame uses a larger sample. Press play, or drag the slider, to move from a small sample to a larger one.
anim = FuncAnimation(fig, draw, frames=len(sizes), interval=500, blit=False)
player = HTML(anim.to_jshtml())
# Close the static figure so only the player is shown.
plt.close(fig)
# The player under the cell.
player


## Check yourself

1. The recovered least-squares slope for $x = 1, 2, 3$ and $y = 2, 4, 6$. Write that rule as the body of `recovered_slope`.
2. The recovered intercept for $x = 1, 2, 3$ and $y = 3, 5, 7$. Write that rule as the body of `recovered_intercept`.

Type the function body in place of `...` in the cell below, then run the cell. A match prints `That matches.` A first miss prints a hint. Run the cell again, or pass `show=True`, to see the expected result. The cell loads `check` from `check.py` in the file list.

If the blank is inside a function, include `return` so the function gives a value back.


In [ ]:
# Load check from check.py so this cell can compare what you type with the stored result.
# On the notebook page that file is at /drive. Opened from a lesson folder, it is in the parent folder.
import sys
from pathlib import Path

# Use /drive when the file is there. Otherwise use the parent folder.
if Path("/drive/check.py").is_file():
    root = "/drive"
elif (Path.cwd().parent / "check.py").is_file():
    root = str(Path.cwd().parent)
else:
    root = ""
# Add that folder to the places Python searches, then load check.
if root and root not in sys.path:
    sys.path.append(root)
from check import check

def recovered_slope(xs, ys):
    # Question 1: type the function body in place of ..., and use return so the function gives a value back. Without return, the function gives back nothing.
    ...

def recovered_intercept(xs, ys):
    # Question 2: type the function body in place of ..., and use return so the function gives a value back. Without return, the function gives back nothing.
    ...

# Run this check. A match prints That matches. A first miss prints the hint.
check("cm05-1", recovered_slope([1, 2, 3], [2, 4, 6]), hint="Use the least-squares slope from this lesson.")
# Run this check. A match prints That matches. A first miss prints the hint.
check("cm05-2", recovered_intercept([1, 2, 3], [3, 5, 7]), hint="Mean of y, minus the slope times the mean of x.")


## What you practised

You fitted the line the points sat on, and the intercept and slope came back as the ones that had made the line. After one score moved, the fit followed the points in front of it.
